# Synthetic CGM Readings API (prototype)

This notebook runs a local, device-neutral API and exercises it with synthetic data. It does not connect to a CGM, provide clinical interpretation, or accept real patient data. Do not expose this unauthenticated prototype to the internet.


In [ ]:
!git clone https://github.com/contrivancecompanychicago/pirddiiqlabdecisions.git
%cd pirddiiqlabdecisions
!git checkout feature/chicago-cgm-access
!python -m pip install -e .

In [ ]:
import os
import subprocess
import sys
import time
import requests
import pandas as pd

os.environ['PIRDDIIQLAB_CGM_DB_PATH'] = '/content/cgm_synthetic_demo.sqlite3'
api_process = subprocess.Popen([sys.executable, '-m', 'pirddiiqlab.cgm_api'], env=os.environ.copy())
base_url = 'http://127.0.0.1:5000'
for _ in range(30):
    try:
        if requests.get(f'{base_url}/health', timeout=1).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('CGM demo API did not start')
print(requests.get(f'{base_url}/health').json())

## Load and submit the synthetic readings

The CSV contains 30 invented readings for three synthetic participant IDs. Re-running the POST cell inserts another demo batch.

In [ ]:
dataset = pd.read_csv('data/synthetic_cgm_readings.csv')
payload = {'readings': dataset.to_dict(orient='records')}
response = requests.post(f'{base_url}/api/v1/cgm/readings', json=payload, timeout=10)
print(response.status_code, response.json())
response.raise_for_status()
assert response.json()['count'] == len(dataset)

## Retrieve one participant's readings

The API requires a participant ID on reads to avoid returning an unfiltered full dataset. Values shown are raw synthetic readings, not clinical advice.

In [ ]:
result = requests.get(
    f'{base_url}/api/v1/cgm/readings',
    params={'participant_id': 'synthetic-001', 'limit': 100},
    timeout=10,
).json()
print('Count:', result['count'])
pd.DataFrame(result['readings']).head()

## Exercise validation and run automated tests

The following checks validate API input behavior only; they do not validate device accuracy or clinical safety.

In [ ]:
bad = requests.post(
    f'{base_url}/api/v1/cgm/readings',
    json={'readings': [{'participant_id': 'synthetic-001', 'timestamp': 'no-timezone', 'glucose_mg_dl': 100}]},
    timeout=10,
)
assert bad.status_code == 400, bad.text
print('Invalid timestamp correctly rejected:', bad.json())
!python -m pytest tests/test_cgm_api.py -q

## Stop the local demo server

Do not upload identifiable health data to this notebook or any hosted notebook environment.

In [ ]:
api_process.terminate()
api_process.wait(timeout=5)
print('Demo API stopped')